In [ ]:
# ====================== GPU PICKER (server-friendly) ======================
# Set which GPU to use (0, 1, ...). Set to None to NOT force anything (respects the environment).
GPU_ID =   1 # Change to None when you want to leave it free for other users
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
torch.backends.cudnn.benchmark = True  # if the size varies A LOT, consider False
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.set_device(0)
    torch.cuda.set_per_process_memory_fraction(0.45, device=0)
print("Device:", device, "| Visible devices (after mapping):", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if device == "cuda":
    try:
        print("Using mapped GPU:", torch.cuda.get_device_name(0))
    except Exception:
        pass
    for i in range(torch.cuda.device_count()):
        try:
            print(f"Device {i}:", torch.cuda.get_device_name(i))
        except Exception:
            pass
import sys
from datetime import datetime
import time

In [ ]:
import sys
sys.path.insert(0, "/workspace/app")
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import importlib
from coding.Data_Processing import Run_RNN as rf
importlib.reload(rf)
from coding.Data_Preprocessing import DataPreprocessing_melt as mf
from coding.Data_Processing import TargetVariables as tv
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90  

___

LOG

In [ ]:
class Tee(object):
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

In [ ]:
log_path = f"COLA_REG_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
orig_stdout = sys.stdout
orig_stderr = sys.stderr
log_file = open(log_path, "w", buffering=1) 
sys.stdout = Tee(orig_stdout, log_file)
sys.stderr = Tee(orig_stderr, log_file)
start_time = time.perf_counter()
print("---- Logging started ----")
print("Log file:", log_path)

___

Dataset

In [ ]:
df_wav = pd.read_pickle('/workspace/app/planilhas/df_emb3D.pkl') # wav2vec
df_wav['Patient_ID'] = encoder.fit_transform(df_wav['Patient_ID'])
print(f"Patients: {df_wav['Patient_ID'].nunique()}")

In [ ]:
df_wav = tv.standardized_T1(df_wav, MAX_HDRS, MAX_CDI)
df_wav["Y_Standardized_T1"] = pd.to_numeric(df_wav["Y_Standardized_T1"], errors="coerce")

___

Preprocessing

In [ ]:
suffixes = ('Date', 'F1_Score', 'F2_Score', 'F3_Score', '_Days', 'soc_Score', 'iso_Score', 'sup_Score')
prefixes = ('Questionnary1', 'Questionnary2', 'LogMel_Pat_Speech', 'Days_')
exact_cols = {
    "Age", 'Patient_Gender', 'Randomization_Group', 'Education', 'Number_Complete_Sessions',
    'Questionnary5_HQ_25_T0_TOT_Score', 'Questionnary5_HQ_25_T1_TOT_Score',
    'Questionnary3_TAS_20_T1_TOT_Score', 'Questionnary4_AQC_T1_TOT_Score'} # Alexithymia T1 to avoid leakage
cols_to_drop = [
    col for col in df_wav.columns 
    if col in exact_cols or col.endswith(suffixes) or col.startswith(prefixes)]
df_wav.drop(columns=cols_to_drop, inplace=True)

In [ ]:
def get_embeddings_per_segment_mean_3D(df, meta_cols):
    embedding_cols = [
        col for col in df.columns
        if col.startswith("Embeddings_Session")
    ]

    if not embedding_cols:
        raise ValueError("No Embeddings_Session columns were found.")

    df_long = df.melt(id_vars=meta_cols, value_vars=embedding_cols, var_name="Embedding_feature", value_name="Embedding_npy")

    extracted = df_long["Embedding_feature"].str.extract(r"Embeddings_Session(\d+)_Segment(\d+)")
    df_long["Session"] = pd.to_numeric(extracted[0], errors="coerce")
    df_long["Segment"] = pd.to_numeric(extracted[1], errors="coerce")

    df_long = df_long.dropna(subset=["Session", "Segment", "Embedding_npy"]).copy()

    df_long["Session"] = df_long["Session"].astype(int)
    df_long["Segment"] = df_long["Segment"].astype(int)

    new_order = ["Patient_ID", "Session", "Segment"]
    new_order += [col for col in meta_cols if col != "Patient_ID"]
    new_order += ["Embedding_npy"]

    df_long = df_long[new_order].reset_index(drop=True)

    duplicated = df_long.duplicated(subset=["Patient_ID", "Session", "Segment"])
    if duplicated.any():
        raise ValueError(f"{duplicated.sum()} duplicated patient/session/segment rows found.")
    return df_long

In [ ]:
meta = ['Patient_ID', 'Questionnary3_TAS_20_T0_TOT_Score', 'Questionnary4_AQC_T0_TOT_Score', 'Y_Standardized_T1']
df_wav = get_embeddings_per_segment_mean_3D(df_wav, meta)

In [ ]:
df_cnn = pd.read_pickle("/workspace/app/planilhas/df_cnn_segments_reg.pkl")

___

### Experiments

In [ ]:
unique_patients = df_cnn['Patient_ID'].unique()
target = "Y_Standardized_T1"
alex_col="Alexithymia_T0"
num_epochs = 20
BS = 2
lambda_alex = 0.05
# beta_kl = 0.0
# stoc = False

#### Experiment 2 
a) Alexithymia-Query Conditioned Longitudinal Attention (AQ-CoLA) - Regression usng CNN-log
- mode="full"
- trait_source="alex" | Standardized alexithymia within the fold enters the network, is transformed into a z-trait, and participates in the attention query

The alexithymia used in this variant should be the Alexithymia_T0 constructed with mean and standard deviation calculated only in the training patients of each fold.


* obs.: For trait_source="speech":  The query comes from the acoustic representation of the first session. Alexithymia is used only as an auxiliary supervision tool. - LOSS

In [ ]:
""" print(f"Processing Dataset COLA_Alex_LogMel_Full Regression")
results = [
        rf.reg_lopoRNN(
                p, df_cnn, 'Patient_ID', 'CNN_SegEmb_npy', target, model_name = "COLA", alex_col=alex_col,
                modeCola = "full", representation="cnn", num_epochs=num_epochs, BS=BS, trait_source="alex")
        for p in unique_patients]
print("COLA_Alex_LogMel_Full Done")
df_results = pd.DataFrame(results)
df_summary = pd.DataFrame([{
        "Model": "COLA_Alex_LogMel_Full",
        "RMSE_Mean": df_results["RMSE"].mean(),
        "RMSE_Std": df_results["RMSE"].std(),
        "MSE_Mean": df_results["MSE"].mean(),
        "MAE_Mean": df_results["MAE"].mean()
}])
df_results.to_excel('/workspace/app/planilhas/TaCoLa/REG_COLA_Alex_LogMel_Full.xlsx', index=False)
#df_summary.to_excel('/workspace/app/planilhas/TaCoLa/REG_COLA_Alex_LogMel_Full_SUMMARY.xlsx', index=False) """

b) Alexithymia-Query Conditioned Longitudinal Attention (AQ-CoLA) - Regression usng wav2vec

In [ ]:
""" print("Processing Dataset COLA_Alex_Wav_Full Regression")
results = [
    rf.reg_lopoRNN(
        p, df_wav, "Patient_ID", "Embedding_npy", target, model_name="COLA", alex_col=alex_col, 
        modeCola="full", representation="wav2vec", num_epochs=num_epochs, BS=BS,
        device=device, trait_source="alex", lambda_alex=lambda_alex)
    for p in unique_patients]
print("COLA_Alex_Wav_Full Done")
df_results = pd.DataFrame(results)
df_summary = pd.DataFrame([{
        "Model": "COLA_Alex_Wav_Full",
        "RMSE_Mean": df_results["RMSE"].mean(),
        "RMSE_Std": df_results["RMSE"].std(),
        "MSE_Mean": df_results["MSE"].mean(),
        "MAE_Mean": df_results["MAE"].mean()
}])
df_results.to_excel("/workspace/app/planilhas/TaCoLa/REG_COLA_Alex_Wav_Full.xlsx", index=False)
df_summary.to_excel("/workspace/app/planilhas/TaCoLa/REG_COLA_Alex_Wav_Full_SUMMARY.xlsx", index=False) """

___

#### Experiment 3 - CoLA Ablations (Regression)
Representation: Use the best from Experiment 2 (CNN-log or wav2vec)

1) Ablations of the CoLA approach

trait_source = "alex"

Compare:
* trait:       [h, z]
* interaction: [h, z, h*z]
* delta_h1:    [h, z, h-h1]
* full:        [h, z, h*z, h-h1]

In [ ]:
""" # Use the best representation from Experiment 2 (CNN-log or wav2vec)
dic_path = {
    "path_cnn": '/workspace/app/planilhas/REG_COLA_Alex_LogMel_Full_SUMMARY.xlsx', 
    "path_wav": '/workspace/app/planilhas/REG_COLA_Alex_Wav_Full_SUMMARY.xlsx'
} 
if os.path.isfile(dic_path["path_cnn"]) and os.path.isfile(dic_path["path_wav"]):
    df_cnn_SUM = pd.read_excel(dic_path["path_cnn"])
    df_wav_SUM = pd.read_excel(dic_path["path_wav"])
    rmse_cnn = df_cnn_SUM.loc[df_cnn_SUM["Model"] == "COLA_Alex_LogMel_Full", "RMSE_Mean"].values[0]
    rmse_wav = df_wav_SUM.loc[df_wav_SUM["Model"] == "COLA_Alex_Wav_Full", "RMSE_Mean"].values[0]
    if rmse_cnn < rmse_wav:
        best_representation = "cnn"
        best_df = df_cnn.copy()
    else:
        best_representation = "wav2vec"
        best_df = df_wav.copy()
    print(f"Best Representation: {best_representation}") """

In [ ]:
""" representation = best_representation
npy_col = "CNN_SegEmb_npy" if best_representation == "cnn" else "Embedding_npy"
name_best = "COLA_Alex_LogMel" if best_representation == "cnn" else "COLA_Alex_Wav" """

In [ ]:
""" experiments = [
    {"name": f"{name_best}_Trait", "modeCola": "trait"},
    {"name": f"{name_best}_Inter", "modeCola": "interaction"},
    {"name": f"{name_best}_DELTAH1", "modeCola": "delta_h1"}
]
for exp in experiments:
    print(f"Processing Dataset {exp['name']} Regression")
    results = [
        rf.reg_lopoRNN(
            p, best_df, "Patient_ID", npy_col, target, model_name="COLA", alex_col=alex_col, 
            modeCola=exp["modeCola"], trait_source="alex", lambda_alex=lambda_alex,
            representation=best_representation, num_epochs=num_epochs, BS=BS, device=device)
        for p in unique_patients]
    print(f"{exp['name']} Done")
    df_results = pd.DataFrame(results)
    df_summary = pd.DataFrame([{
        "Model": exp["name"],
        "RMSE_Mean": df_results["RMSE"].mean(),
        "RMSE_Std": df_results["RMSE"].std(),
        "MSE_Mean": df_results["MSE"].mean(),
        "MAE_Mean": df_results["MAE"].mean()
    }])
    df_results.to_excel(f'/workspace/app/planilhas/REG_{exp["name"]}.xlsx', index=False)
    df_summary.to_excel(f'/workspace/app/planilhas/REG_{exp["name"]}_SUMMARY.xlsx', index=False) """

Seed

In [ ]:
for seed in [42, 123, 2026]:
    print(f"REG_COLA_Alex_LogMel_Trait | seed={seed}")
    results_reg = [
        rf.reg_lopoRNN(
            p, df_cnn, "Patient_ID", "CNN_SegEmb_npy", target, model_name="COLA",
            alex_col=alex_col, modeCola="trait", representation="cnn",
            num_epochs=num_epochs, BS=BS, device=device, seed=seed,
            trait_source="alex", lambda_alex=lambda_alex)
        for p in unique_patients
    ]
    df_results = pd.DataFrame(results_reg)
    df_results["Seed"] = seed
    df_results["Experiment"] = "COLA_Alex_LogMel_Trait"
    df_results.to_excel(f"/workspace/app/planilhas/TaCoLa/Seeds/REG_COLA_Alex_LogMel_Trait_seed{seed}.xlsx", index=False)
    print(f"REG_COLA_Alex_LogMel_Trait | seed={seed} Done")

___

2) Trait Origin
Fix the best mode and compare:

trait_source="alex"

trait_source="speech"

* This answers:
Is measured baseline alexithymia more useful than a speech-derived latent trait supervised by alexithymia?

This comparison separates two truly different hypotheses.

In [ ]:
""" best_modeCola = "full"  # Replace after selecting the best ablation
print("Processing Dataset COLA_Speech Regression")
results = [
    rf.reg_lopoRNN(
        p, best_df, "Patient_ID", npy_col, target, model_name="COLA", alex_col=alex_col, 
        modeCola=best_modeCola, trait_source="speech", lambda_alex=lambda_alex,
        representation=best_representation, num_epochs=num_epochs, BS=BS, device=device)
    for p in unique_patients]
print("COLA_Speech Done")
df_results = pd.DataFrame(results)
df_summary = pd.DataFrame([{
    "Model": "COLA_Speech_REG",
    "RMSE_Mean": df_results["RMSE"].mean(),
    "RMSE_Std": df_results["RMSE"].std(),
    "MSE_Mean": df_results["MSE"].mean(),
    "MAE_Mean": df_results["MAE"].mean()
}])
df_results.to_excel("/workspace/app/planilhas/TaCoLa/REG_COLA_Speech.xlsx", index=False)
df_summary.to_excel("/workspace/app/planilhas/TaCoLa/REG_COLA_Speech_SUMMARY.xlsx", index=False) """

3. [ON HOLD] Variational, only if justified

If trait_source="speech" works:

deterministic speech versus stochastic variational speech

* mu_layer | logvar_layer | reparameterize | beta_kl

___

LOG

In [ ]:
elapsed_time = time.perf_counter() - start_time
print("\n---- Experiment finished ----")
print(f"Total execution time: {elapsed_time:.2f} seconds")
print(f"Total execution time: {elapsed_time / 60:.2f} minutes")